In [1]:
import sys
from tqdm import tqdm
import numpy as np
import os
import pandas as pd
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score
from torch.utils.data import Dataset
import time
from torch.utils.data import DataLoader
import json
from sklearn.model_selection import train_test_split
import argparse
from torch import optim
import pandas as pd
import numpy as np
import os
# import seaborn as sns
from tqdm.auto import tqdm
import warnings
from sklearn.preprocessing import OneHotEncoder
import gc
import pickle
from sklearn.decomposition import TruncatedSVD
import glob
from torch.nn.utils.rnn import pad_sequence
import math
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv
import torch_geometric
from sklearn.calibration import LabelEncoder
import ast

DATA_PATH = r"C:\Coding\mabe\data"
CACHE_PATH = r"C:\Coding\mabe\MABe_2022_TVAE\cache"
SPECIAL_VALUE = -99999.0
CHUNK_SIZE = 1000
SUB_SEQ_LENGTH = 21
SLIDING_WINDOW = 1
ALPHA = 10

True
NVIDIA GeForce RTX 4070 SUPER


In [2]:
import pandas as pd

# Your DataFrame (assume it's called df)

# Step 1: Define the priority mapping
priority_map = {
    'top_left': [
        ('ear_left', 1),
        ('forepaw_left', 2),
        ('headpiece_bottombackleft', 3),
        ('headpiece_bottomfrontleft', 4),
        ('headpiece_topbackleft', 5),
        ('headpiece_topfrontleft', 6),
        ('lateral_left', 7),
    ],
    'top_right': [
        ('ear_right', 1),
        ('forepaw_right', 2),
        ('headpiece_bottombackright', 3),
        ('headpiece_bottomfrontright', 4),
        ('headpiece_topbackright', 5),
        ('headpiece_topfrontright', 6),
        ('lateral_right', 7),
    ],
    'bottom': [
        ('tail_base', 1),
        ('tail_midpoint', 2),
        ('tail_middle_1', 3),
        ('tail_middle_2', 4),
        ('hindpaw_left', 5),
        ('hindpaw_right', 6),
        ('hip_left', 7),
        ('hip_right', 8),
        ('tail_tip', 9),
    ],
    'top_center': [
        ('nose', 1),
        ('head', 2),
        ('neck', 3),
        ('body_center', 4),
    ],
}

# Flatten for quick lookup
bodypart_to_target = {}
for target, parts in priority_map.items():
    for part, priority in parts:
        bodypart_to_target[part] = (target, priority)

# Step 2: Filter only relevant bodyparts (ignore 'spine_1', 'spine_2', etc.)
def reassign_points(path, start, stop):
    df = pd.read_parquet(path)
    # df['video_frame'] = df['video_frame'] - df['video_frame'].iloc[0]
    # print(f"shape before: {df.shape}")
    # Filter early: only keep frames within the desired range
    df = df[(df['video_frame'] >= start) & (df['video_frame'] < stop)]
    # print(f"shape after: {df.shape}")

    # df['assignment'] = df['bodypart'].map(bodypart_to_target)

    # # Drop rows with no assignment
    # df = df.dropna(subset=['assignment'])

    # # Add target and priority columns
    # df[['target', 'priority']] = pd.DataFrame(df['assignment'].tolist(), index=df.index)

    # # Keep only the highest priority (lowest number) per group, per target
    # df_sorted = df.sort_values(by=['video_frame', 'mouse_id', 'target', 'priority'])
    # df_deduped = df_sorted.drop_duplicates(subset=['video_frame', 'mouse_id', 'target'], keep='first')

    # Pivot to get final structure
    pivot_x = df.pivot(index='video_frame', columns=['mouse_id', 'bodypart'], values='x')
    pivot_y = df.pivot(index='video_frame', columns=['mouse_id', 'bodypart'], values='y')
    pivot_x.columns = [f"mouse_{m}_{bp}_x" for m, bp in pivot_x.columns]
    pivot_y.columns = [f"mouse_{m}_{bp}_y" for m, bp in pivot_y.columns]
    df_wide = pd.concat([pivot_x, pivot_y], axis=1).sort_index(axis=1)
    # print(df_final.shape)

    return df_wide

In [3]:
body_parts = [
    "ear_left",
    "ear_right",
    "tail_base",
    "nose",
    "neck",
    # "body_center",
    # "tail_tip",
    # "tail_midpoint",
    # "forepaw_left",
    # "forepaw_right",
    # "hindpaw_left",
    # "hindpaw_right",
    "hip_left",
    "hip_right",
    # "lateral_left",
    # "lateral_right",
    # "spine_1",
    # "spine_2",
    # "tail_middle_1",
    # "tail_middle_2",
    # "head",
    # "headpiece_bottombackleft",
    # "headpiece_bottombackright",
    # "headpiece_bottomfrontleft",
    # "headpiece_bottomfrontright",
    # "headpiece_topbackleft",
    # "headpiece_topbackright",
    # "headpiece_topfrontleft",
    # "headpiece_topfrontright"
]
# body_parts = ['top_center', 'top_left', 'top_right', 'bottom']
len(body_parts)

all_cols = []
for i in range(1, 5):
    for part in body_parts:
        for j in ['x', 'y']:
            all_cols.append(f"mouse_{i}_{part}_{j}")
print(all_cols)
final_cols = ['mouse_1_ear_left_x', 'mouse_1_ear_left_y', 'mouse_1_ear_right_x', 'mouse_1_ear_right_y', 'mouse_1_tail_base_x', 'mouse_1_tail_base_y', 'mouse_1_nose_x', 'mouse_1_nose_y', 'mouse_1_neck_x', 'mouse_1_neck_y', 'mouse_1_hip_left_x', 'mouse_1_hip_left_y', 'mouse_1_hip_right_x', 'mouse_1_hip_right_y', 'mouse_2_ear_left_x', 'mouse_2_ear_left_y', 'mouse_2_ear_right_x', 'mouse_2_ear_right_y', 'mouse_2_tail_base_x', 'mouse_2_tail_base_y', 'mouse_2_nose_x', 'mouse_2_nose_y', 'mouse_2_neck_x', 'mouse_2_neck_y', 'mouse_2_hip_left_x', 'mouse_2_hip_left_y', 'mouse_2_hip_right_x', 'mouse_2_hip_right_y', 'mouse_3_ear_left_x', 'mouse_3_ear_left_y', 'mouse_3_ear_right_x', 'mouse_3_ear_right_y', 'mouse_3_tail_base_x', 'mouse_3_tail_base_y', 'mouse_3_nose_x', 'mouse_3_nose_y', 'mouse_3_neck_x', 'mouse_3_neck_y', 'mouse_3_hip_left_x', 'mouse_3_hip_left_y', 'mouse_3_hip_right_x', 'mouse_3_hip_right_y', 'mouse_4_ear_left_x', 'mouse_4_ear_left_y', 'mouse_4_ear_right_x', 'mouse_4_ear_right_y', 'mouse_4_tail_base_x', 'mouse_4_tail_base_y', 'mouse_4_nose_x', 'mouse_4_nose_y', 'mouse_4_neck_x', 'mouse_4_neck_y', 'mouse_4_hip_left_x', 'mouse_4_hip_left_y', 'mouse_4_hip_right_x', 'mouse_4_hip_right_y', 'mouse1_nose_speed', 'mouse1_ear_left_speed', 'mouse1_ear_right_speed', 'mouse1_neck_speed', 'mouse1_tail_base_speed', 'mouse2_nose_speed', 'mouse2_ear_left_speed', 'mouse2_ear_right_speed', 'mouse2_neck_speed', 'mouse2_tail_base_speed', 'mouse3_nose_speed', 'mouse3_ear_left_speed', 'mouse3_ear_right_speed', 'mouse3_neck_speed', 'mouse3_tail_base_speed', 'mouse4_nose_speed', 'mouse4_ear_left_speed', 'mouse4_ear_right_speed', 'mouse4_neck_speed', 'mouse4_tail_base_speed', 'dist_m1nose_m2nose', 'dist_m1nose_m2ear_left', 'dist_m1nose_m2ear_right', 'dist_m1nose_m2neck', 'dist_m1nose_m2tail_base', 'dist_m1ear_left_m2nose', 'dist_m1ear_left_m2ear_left', 'dist_m1ear_left_m2ear_right', 'dist_m1ear_left_m2neck', 'dist_m1ear_left_m2tail_base', 'dist_m1ear_right_m2nose', 'dist_m1ear_right_m2ear_left', 'dist_m1ear_right_m2ear_right', 'dist_m1ear_right_m2neck', 'dist_m1ear_right_m2tail_base', 'dist_m1neck_m2nose', 'dist_m1neck_m2ear_left', 'dist_m1neck_m2ear_right', 'dist_m1neck_m2neck', 'dist_m1neck_m2tail_base', 'dist_m1tail_base_m2nose', 'dist_m1tail_base_m2ear_left', 'dist_m1tail_base_m2ear_right', 'dist_m1tail_base_m2neck', 'dist_m1tail_base_m2tail_base', 'dist_m1nose_m3nose', 'dist_m1nose_m3ear_left', 'dist_m1nose_m3ear_right', 'dist_m1nose_m3neck', 'dist_m1nose_m3tail_base', 'dist_m1ear_left_m3nose', 'dist_m1ear_left_m3ear_left', 'dist_m1ear_left_m3ear_right', 'dist_m1ear_left_m3neck', 'dist_m1ear_left_m3tail_base', 'dist_m1ear_right_m3nose', 'dist_m1ear_right_m3ear_left', 'dist_m1ear_right_m3ear_right', 'dist_m1ear_right_m3neck', 'dist_m1ear_right_m3tail_base', 'dist_m1neck_m3nose', 'dist_m1neck_m3ear_left', 'dist_m1neck_m3ear_right', 'dist_m1neck_m3neck', 'dist_m1neck_m3tail_base', 'dist_m1tail_base_m3nose', 'dist_m1tail_base_m3ear_left', 'dist_m1tail_base_m3ear_right', 'dist_m1tail_base_m3neck', 'dist_m1tail_base_m3tail_base', 'dist_m1nose_m4nose', 'dist_m1nose_m4ear_left', 'dist_m1nose_m4ear_right', 'dist_m1nose_m4neck', 'dist_m1nose_m4tail_base', 'dist_m1ear_left_m4nose', 'dist_m1ear_left_m4ear_left', 'dist_m1ear_left_m4ear_right', 'dist_m1ear_left_m4neck', 'dist_m1ear_left_m4tail_base', 'dist_m1ear_right_m4nose', 'dist_m1ear_right_m4ear_left', 'dist_m1ear_right_m4ear_right', 'dist_m1ear_right_m4neck', 'dist_m1ear_right_m4tail_base', 'dist_m1neck_m4nose', 'dist_m1neck_m4ear_left', 'dist_m1neck_m4ear_right', 'dist_m1neck_m4neck', 'dist_m1neck_m4tail_base', 'dist_m1tail_base_m4nose', 'dist_m1tail_base_m4ear_left', 'dist_m1tail_base_m4ear_right', 'dist_m1tail_base_m4neck', 'dist_m1tail_base_m4tail_base', 'dist_m2nose_m3nose', 'dist_m2nose_m3ear_left', 'dist_m2nose_m3ear_right', 'dist_m2nose_m3neck', 'dist_m2nose_m3tail_base', 'dist_m2ear_left_m3nose', 'dist_m2ear_left_m3ear_left', 'dist_m2ear_left_m3ear_right', 'dist_m2ear_left_m3neck', 'dist_m2ear_left_m3tail_base', 'dist_m2ear_right_m3nose', 'dist_m2ear_right_m3ear_left', 'dist_m2ear_right_m3ear_right', 'dist_m2ear_right_m3neck', 'dist_m2ear_right_m3tail_base', 'dist_m2neck_m3nose', 'dist_m2neck_m3ear_left', 'dist_m2neck_m3ear_right', 'dist_m2neck_m3neck', 'dist_m2neck_m3tail_base', 'dist_m2tail_base_m3nose', 'dist_m2tail_base_m3ear_left', 'dist_m2tail_base_m3ear_right', 'dist_m2tail_base_m3neck', 'dist_m2tail_base_m3tail_base', 'dist_m2nose_m4nose', 'dist_m2nose_m4ear_left', 'dist_m2nose_m4ear_right', 'dist_m2nose_m4neck', 'dist_m2nose_m4tail_base', 'dist_m2ear_left_m4nose', 'dist_m2ear_left_m4ear_left', 'dist_m2ear_left_m4ear_right', 'dist_m2ear_left_m4neck', 'dist_m2ear_left_m4tail_base', 'dist_m2ear_right_m4nose', 'dist_m2ear_right_m4ear_left', 'dist_m2ear_right_m4ear_right', 'dist_m2ear_right_m4neck', 'dist_m2ear_right_m4tail_base', 'dist_m2neck_m4nose', 'dist_m2neck_m4ear_left', 'dist_m2neck_m4ear_right', 'dist_m2neck_m4neck', 'dist_m2neck_m4tail_base', 'dist_m2tail_base_m4nose', 'dist_m2tail_base_m4ear_left', 'dist_m2tail_base_m4ear_right', 'dist_m2tail_base_m4neck', 'dist_m2tail_base_m4tail_base', 'dist_m3nose_m4nose', 'dist_m3nose_m4ear_left', 'dist_m3nose_m4ear_right', 'dist_m3nose_m4neck', 'dist_m3nose_m4tail_base', 'dist_m3ear_left_m4nose', 'dist_m3ear_left_m4ear_left', 'dist_m3ear_left_m4ear_right', 'dist_m3ear_left_m4neck', 'dist_m3ear_left_m4tail_base', 'dist_m3ear_right_m4nose', 'dist_m3ear_right_m4ear_left', 'dist_m3ear_right_m4ear_right', 'dist_m3ear_right_m4neck', 'dist_m3ear_right_m4tail_base', 'dist_m3neck_m4nose', 'dist_m3neck_m4ear_left', 'dist_m3neck_m4ear_right', 'dist_m3neck_m4neck', 'dist_m3neck_m4tail_base', 'dist_m3tail_base_m4nose', 'dist_m3tail_base_m4ear_left', 'dist_m3tail_base_m4ear_right', 'dist_m3tail_base_m4neck', 'dist_m3tail_base_m4tail_base']
print(len(final_cols))


['mouse_1_ear_left_x', 'mouse_1_ear_left_y', 'mouse_1_ear_right_x', 'mouse_1_ear_right_y', 'mouse_1_tail_base_x', 'mouse_1_tail_base_y', 'mouse_1_nose_x', 'mouse_1_nose_y', 'mouse_1_neck_x', 'mouse_1_neck_y', 'mouse_1_hip_left_x', 'mouse_1_hip_left_y', 'mouse_1_hip_right_x', 'mouse_1_hip_right_y', 'mouse_2_ear_left_x', 'mouse_2_ear_left_y', 'mouse_2_ear_right_x', 'mouse_2_ear_right_y', 'mouse_2_tail_base_x', 'mouse_2_tail_base_y', 'mouse_2_nose_x', 'mouse_2_nose_y', 'mouse_2_neck_x', 'mouse_2_neck_y', 'mouse_2_hip_left_x', 'mouse_2_hip_left_y', 'mouse_2_hip_right_x', 'mouse_2_hip_right_y', 'mouse_3_ear_left_x', 'mouse_3_ear_left_y', 'mouse_3_ear_right_x', 'mouse_3_ear_right_y', 'mouse_3_tail_base_x', 'mouse_3_tail_base_y', 'mouse_3_nose_x', 'mouse_3_nose_y', 'mouse_3_neck_x', 'mouse_3_neck_y', 'mouse_3_hip_left_x', 'mouse_3_hip_left_y', 'mouse_3_hip_right_x', 'mouse_3_hip_right_y', 'mouse_4_ear_left_x', 'mouse_4_ear_left_y', 'mouse_4_ear_right_x', 'mouse_4_ear_right_y', 'mouse_4_tail_b

In [4]:
# List of mouse IDs and core body parts
mouse_ids = [1, 2, 3, 4]
CORE_BODYPARTS = ['nose', 'ear_left', 'ear_right', 'neck', 'tail_base']
# Start grouping columns by mouse
mouse_columns = {mid: [] for mid in mouse_ids}
interaction_columns = []

for col in final_cols:
    added = False
    for mid in mouse_ids:
        if f'mouse_{mid}_' in col or f'mouse{mid}_' in col:
            mouse_columns[mid].append(col)
            added = True
            break
    if not added and col.startswith('dist_m'):
        interaction_columns.append(col)  # These are pairwise interaction features

# Reconstruct column order: mouse 1 features → mouse 2 features → ... → interaction features
new_column_order = []
for mid in mouse_ids:
    new_column_order.extend(sorted(mouse_columns[mid]))  # Optionally sort each group
# new_column_order.extend(sorted(interaction_columns))  # Sort interaction features last
print(new_column_order)
print(len(new_column_order))

['mouse1_ear_left_speed', 'mouse1_ear_right_speed', 'mouse1_neck_speed', 'mouse1_nose_speed', 'mouse1_tail_base_speed', 'mouse_1_ear_left_x', 'mouse_1_ear_left_y', 'mouse_1_ear_right_x', 'mouse_1_ear_right_y', 'mouse_1_hip_left_x', 'mouse_1_hip_left_y', 'mouse_1_hip_right_x', 'mouse_1_hip_right_y', 'mouse_1_neck_x', 'mouse_1_neck_y', 'mouse_1_nose_x', 'mouse_1_nose_y', 'mouse_1_tail_base_x', 'mouse_1_tail_base_y', 'mouse2_ear_left_speed', 'mouse2_ear_right_speed', 'mouse2_neck_speed', 'mouse2_nose_speed', 'mouse2_tail_base_speed', 'mouse_2_ear_left_x', 'mouse_2_ear_left_y', 'mouse_2_ear_right_x', 'mouse_2_ear_right_y', 'mouse_2_hip_left_x', 'mouse_2_hip_left_y', 'mouse_2_hip_right_x', 'mouse_2_hip_right_y', 'mouse_2_neck_x', 'mouse_2_neck_y', 'mouse_2_nose_x', 'mouse_2_nose_y', 'mouse_2_tail_base_x', 'mouse_2_tail_base_y', 'mouse3_ear_left_speed', 'mouse3_ear_right_speed', 'mouse3_neck_speed', 'mouse3_nose_speed', 'mouse3_tail_base_speed', 'mouse_3_ear_left_x', 'mouse_3_ear_left_y', 'm

In [5]:
def rotate_keypoints(df: pd.DataFrame, width: int, height: int, angle: int) -> pd.DataFrame:
    if angle == 0:
        return df
    if angle not in [90, 180, 270]:
        raise ValueError("Angle must be one of: 90, 180, or 270 degrees")

    df_rot = df.copy()

    # Get all unique keypoint prefixes (like 'mouse_1_ear_left', 'mouse_2_nose', etc.)
    base_names = sorted({col.rsplit('_', 1)[0] for col in df.columns})

    for name in base_names:
        x_col = f"{name}_x"
        y_col = f"{name}_y"

        x = df[x_col]
        y = df[y_col]

        if angle == 90:
            x_new = height - y - 1
            y_new = x
        elif angle == 180:
            x_new = width - x - 1
            y_new = height - y - 1
        elif angle == 270:
            x_new = y
            y_new = width - x - 1

        df_rot[x_col] = x_new
        df_rot[y_col] = y_new

    return df_rot

In [6]:
def normalize(data, FRAME_WIDTH_TOP, FRAME_HEIGHT_TOP):
    """Normalize coordinate data (NumPy array or DataFrame)."""
    is_df = isinstance(data, pd.DataFrame)
    values = data.values if is_df else data
    if values.shape[1] % 2 != 0:
        raise ValueError("Expected even number of columns representing (x, y) coordinate pairs.")

    state_dim = values.shape[1] // 2
    shift = np.array([FRAME_WIDTH_TOP / 2, FRAME_HEIGHT_TOP / 2] * state_dim)
    scale = np.array([FRAME_WIDTH_TOP / 2, FRAME_HEIGHT_TOP / 2] * state_dim)

    normalized = (values - shift) / scale
    # normalized = np.where(np.isnan(values), -1, normalized)

    if is_df:
        return pd.DataFrame(normalized, columns=data.columns, index=data.index)
    else:
        return normalized

def unnormalize(data, FRAME_WIDTH_TOP, FRAME_HEIGHT_TOP):
    """
    Undo normalization of coordinate data.
    Accepts NumPy arrays, Pandas DataFrames, or PyTorch tensors.
    Expects data in the format: [batch_size, x1, y1, x2, y2, ..., xn, yn]
    """

    # Convert torch tensor to NumPy array
    if torch.is_tensor(data):
        data = data.detach().cpu().numpy()

    is_df = isinstance(data, pd.DataFrame)
    values = data.values if is_df else data

    state_dim = values.shape[1] // 2

    x_shift = FRAME_WIDTH_TOP / 2
    y_shift = FRAME_HEIGHT_TOP / 2
    x_scale = FRAME_WIDTH_TOP / 2
    y_scale = FRAME_HEIGHT_TOP / 2

    # Unnormalize x and y coordinates separately
    values[:, ::2] = values[:, ::2] * x_scale + x_shift  # x coordinates
    values[:, 1::2] = values[:, 1::2] * y_scale + y_shift  # y coordinates

    # Return in original format
    if is_df:
        return pd.DataFrame(values, columns=data.columns, index=data.index)
    else:
        return values  # shape remains [batch_size, state_dim * 2]


def rotate(data, center_index):
    # data shape is num_seq x 3 x 10 x 2
    
    data = data.reshape(data.shape[0], 4, 29 ,2)
    mice = [data[:,i,:10,:] for i in range(3)]
    data = np.concatenate(mice, axis=0)

    del mice
    gc.collect()

    mouse_center = data[:, center_index, :]
    centered_data = data - mouse_center[:, np.newaxis, :]

	# Rotate such that keypoints 3 and 6 are parallel with the y axis
    mouse_rotation = np.arctan2(
		data[:, 3, 0] - data[:, 9, 0], data[:, 3, 1] - data[:, 9, 1])

    R = (np.array([[np.cos(mouse_rotation), -np.sin(mouse_rotation)],
				   [np.sin(mouse_rotation),  np.cos(mouse_rotation)]]).transpose((2, 0, 1)))

	# Encode mouse rotation as sine and cosine
    mouse_rotation = np.concatenate([np.sin(mouse_rotation)[:, np.newaxis], np.cos(
		mouse_rotation)[:, np.newaxis]], axis=-1)

    centered_data = np.matmul(R, centered_data.transpose(0, 2, 1))
    centered_data = centered_data.transpose((0, 2, 1))
    centered_data = centered_data.reshape((-1, 20))

    return centered_data, mouse_center, mouse_rotation

In [7]:
def load_and_process_video(video_id, lab_id, start, end, width, height, angle, data_path=DATA_PATH):
    # print(f"doing from {start} to {end}")
    tracking_path = os.path.join(data_path, 'train_tracking', lab_id, f'{video_id}.parquet')
    if not os.path.exists(tracking_path):
        return None
    df = reassign_points(tracking_path, start, end)
    # df.set_index(['video_frame', 'mouse_id'], inplace=True)

    # # Step 2: Flatten the structure by pivoting mouse_id into columns
    # df_wide = df.unstack(level='mouse_id')

    # # Step 3: Flatten the MultiIndex column names
    # df_wide.columns = [f'mouse_{mouse_id}_{col}' for col, mouse_id in df_wide.columns]

    df_wide = df.reindex(columns=all_cols)
    df_wide = rotate_keypoints(df_wide, width, height, angle)
    if angle in [90, 270]:
        df_wide = normalize(df_wide, height, width)
    else:
        df_wide = normalize(df_wide, width, height)
    # df_wide = unnormalize(df_wide, width, height)
    df_wide = create_hybrid_features(df_wide)
    df_wide = df_wide.reindex(columns=new_column_order)
    # df_wide.to_csv('output.csv', index=False)

    # For debugging
    # print(f"df long cols {list(df_wide.columns)}")
    # print(f"df long {df_wide.head}")
    # print(f"df wide shape {df_wide.shape}")
    return df_wide

CORE_BODYPARTS = ['nose', 'ear_left', 'ear_right', 'neck' 'tail_base']
def create_advanced_features(df_wide):
    """
    Creates a rich set of kinematic, interaction, and postural features.
    """
    # Start with a copy of the original data
    features_df = df_wide.copy()
    
    mouse_ids = [1, 2, 3, 4] # Assuming up to 4 mice
    
    # --- 1. Kinematic Features (Speeds) ---
    for mid in mouse_ids:
        for part in CORE_BODYPARTS:
            col_x, col_y = f'mouse_{mid}_{part}_x', f'mouse_{mid}_{part}_y'
            if col_x in features_df.columns:
                delta_x = features_df[col_x].diff()
                delta_y = features_df[col_y].diff()
                features_df[f'mouse{mid}_{part}_speed'] = np.sqrt(delta_x**2 + delta_y**2)

    # --- 2. Postural Features (Body Elongation) ---
    for mid in mouse_ids:
        nose_x, nose_y = f'mouse_{mid}_top_center_x', f'mouse_{mid}_top_center_y'
        tail_x, tail_y = f'mouse_{mid}_bottom_x', f'mouse_{mid}_bottom_y'
        if all(c in features_df.columns for c in [nose_x, nose_y, tail_x, tail_y]):
            features_df[f'mouse{mid}_elongation'] = np.sqrt(
                (features_df[nose_x] - features_df[tail_x])**2 + 
                (features_df[nose_y] - features_df[tail_y])**2
            )

    # --- 3. Interaction Features (Distances) ---
    mouse_pairs = [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
    for m1, m2 in mouse_pairs:
        for part1 in CORE_BODYPARTS:
            for part2 in CORE_BODYPARTS:
                p1_x, p1_y = f'mouse_{m1}_{part1}_x', f'mouse_{m1}_{part1}_y'
                p2_x, p2_y = f'mouse_{m2}_{part2}_x', f'mouse_{m2}_{part2}_y'
                if all(c in features_df.columns for c in [p1_x, p1_y, p2_x, p2_y]):
                    features_df[f'dist_m{m1}{part1}_m{m2}{part2}'] = np.sqrt(
                        (features_df[p1_x] - features_df[p2_x])**2 + 
                        (features_df[p1_y] - features_df[p2_y])**2
                    )
                    
    # Drop the original coordinate columns to force the model to use our new features
    features_df = features_df.copy()
    features_df = features_df.drop(columns=df_wide.columns)
    
    return features_df

def create_hybrid_features(df_wide):
    engineered_features = create_advanced_features(df_wide.copy())
    
    hybrid_features = pd.concat([df_wide, engineered_features], axis=1)
    return hybrid_features

In [8]:
remapping_dict = {
    "rear": "rear",
    "avoid": "avoid",
    "attack": "attack",
    "approach": "approach",
    "submit": "no_behavior",
    "chaseattack": "chase",
    "chase": "chase",
    "shepherd": "shepherd",
    "sniff": "sniff",
    "mount": "mount",
    "disengage": "disengage",
    "selfgroom": "selfgroom",
    "sniffgenital": "sniff",
    "sniffbody": "sniff",
    "sniffface": "sniff",
    "dominancemount": "mount",
    "attemptmount": "mount",
    "intromit": "no_behavior",
    "genitalgroom": "selfgroom",
    "reciprocalsniff": "sniff",
    "escape": "escape",
    "dominance": "dominance",
    "allogroom": "no_behavior",
    "ejaculate": "no_behavior",
    "defend": "defend",
    "dig": "dig",
    "rest": "rest",
    "climb": "climb",
    "run": "no_behavior",
    "dominancegroom": "no_behavior",
    "freeze": "no_behavior",
    "follow": "follow",
    "flinch": "no_behavior",
    "biteobject": "object",
    "exploreobject": "object",
    "tussle": "no_behavior",
    "huddle": "huddle"
}

In [9]:
dir_path = os.path.join(DATA_PATH, "train_annotation")

# Recursively find all .parquet files
ann_files = []
for root, _, files in os.walk(dir_path):
    for file in files:
        if file.endswith(".parquet"):
            ann_files.append(os.path.join(root, file))

# Read and concatenate all parquet files into one DataFrame
dfs = []
for f in ann_files:
    df = pd.read_parquet(f)
    fn = os.path.splitext(os.path.basename(f))[0]  # filename without extension
    df['video_file'] = fn
    dfs.append(df)

train_ann = pd.concat(dfs, ignore_index=True)

print(f"Number of files: {len(ann_files)}")
# print(train_ann.head(10))
self_actions = train_ann.loc[train_ann['agent_id'] == train_ann['target_id'], 'action'].map(remapping_dict).unique()

# print("Self actions:", ", ".join(self_actions))

# Pair actions: where agent_id != target_id, unique actions
pair_actions = train_ann.loc[train_ann['agent_id'] != train_ann['target_id'], 'action'].map(remapping_dict).unique()

# print("\nPair actions:", ", ".join(pair_actions))
all_actions = np.unique(np.concatenate((self_actions, pair_actions)))
all_actions = list(all_actions)
self_actions = list(self_actions)
pair_actions = list(pair_actions)
all_actions.append("no_behavior")
self_actions.insert(0, self_actions.pop(self_actions.index("no_behavior")))
pair_actions.insert(0, pair_actions.pop(pair_actions.index("no_behavior")))


solo_encoder = LabelEncoder()
solo_encoder.classes_ = np.array(self_actions)
dual_encoder = LabelEncoder()
dual_encoder.classes_ = np.array(pair_actions)
del train_ann, ann_files

agents = [1, 2, 3, 4]
agent_encoder = LabelEncoder()
agent_encoder.fit(np.array(agents).reshape(-1, 1))
print(len(self_actions), len(pair_actions))
print(pair_actions)

Number of files: 847
8 13
['no_behavior', 'chase', 'avoid', 'attack', 'approach', 'shepherd', 'sniff', 'mount', 'escape', 'disengage', 'dominance', 'defend', 'follow']


c:\Users\PC\.conda\envs\reg\lib\site-packages\sklearn\preprocessing\_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [10]:
def build_chunks_test(path, train_df_path, all_files, chunk_size=CHUNK_SIZE, sub_seq_length=SUB_SEQ_LENGTH):
    chunks = []
    print(f"building test chunks with {len(all_files)} files")

    train_df = pd.read_csv(train_df_path)
    for _, row in train_df.iterrows():
        video_path = os.path.join(path, 'train_annotation', row['lab_id'], f"{row['video_id']}.parquet")
        if video_path not in all_files:
            continue

        df = pd.read_parquet(video_path)

        start_frame = df['start_frame'].iloc[0]
        stop_frame = df['stop_frame'].iloc[-1]

        for start_idx in range(start_frame, stop_frame + 1, chunk_size):
            start_frame2 = max(start_frame, start_idx - sub_seq_length // 2)
            chunk_end = min(start_idx + chunk_size + sub_seq_length // 2, stop_frame)
            chunks.append({
                'video_id': row['video_id'],
                'lab_id': str(row['lab_id']),
                'width': row['video_width_pix'],
                'height': row['video_height_pix'],
                'chunk_start': start_frame2,
                'chunk_end': chunk_end,
                'first_chunk': start_idx == start_frame,
                'last_chunk': chunk_end == stop_frame,
                'angle': 0
            })
    return chunks

def build_chunks(path, train_df_path, all_files, sub_seq_length=SUB_SEQ_LENGTH, min_chunk_len=22, max_chunk_len=2500):
    chunks = []
    print(f"Building chunks with {len(all_files)} files")

    train_df = pd.read_csv(train_df_path)

    for _, row in train_df.iterrows():
        video_path = os.path.join(path, 'train_annotation', str(row['lab_id']), f"{row['video_id']}.parquet")
        if video_path not in all_files:
            continue

        df = pd.read_parquet(video_path)
        df = df.sort_values('start_frame').reset_index(drop=True)

        final_frame = int(df['stop_frame'].iloc[-1])

        # --- 1️⃣ Behavior chunks ---
        for _, ann in df.iterrows():
            start_frame = int(ann['start_frame'])
            stop_frame = int(ann['stop_frame'])
            action = remapping_dict[ann['action']]
            length = stop_frame - start_frame + 1

            if length < min_chunk_len or length > max_chunk_len or action == "no_behavior":
                continue

            chunks.append({
                'video_id': row['video_id'],
                'lab_id': str(row['lab_id']),
                'width': row['video_width_pix'],
                'height': row['video_height_pix'],
                'chunk_start': start_frame,
                'chunk_end': stop_frame,
                'label': remapping_dict[ann['action']],
                'agent_id': ann['agent_id'],
                'target_id': ann['target_id'],
                'first_chunk': start_frame < 10,
                'last_chunk': stop_frame == final_frame,
                'angle': 0,
            })

        # --- 2️⃣ No-behavior chunks ---
        # prev_stop = 0
        # for _, ann in df.iterrows():
        #     start_frame = int(ann['start_frame'])
        #     if start_frame - prev_stop - 1 >= min_chunk_len:
        #         chunks.append({
        #             'video_id': row['video_id'],
        #             'lab_id': str(row['lab_id']),
        #             'width': row['video_width_pix'],
        #             'height': row['video_height_pix'],
        #             'chunk_start': prev_stop + 1,
        #             'chunk_end': start_frame - 1,
        #             'label': 'no_behavior',
        #             'agent_id': None,
        #             'target_id': None,
        #             'first_chunk': prev_stop < 10,
        #             'last_chunk': False,
        #             'angle': 0,
        #         })
        #     prev_stop = int(ann['stop_frame'])

        # # --- 3️⃣ Tail-end no-behavior chunk ---
        # if final_frame - prev_stop >= min_chunk_len:
        #     chunks.append({
        #         'video_id': row['video_id'],
        #         'lab_id': str(row['lab_id']),
        #         'width': row['video_width_pix'],
        #         'height': row['video_height_pix'],
        #         'chunk_start': prev_stop + 1,
        #         'chunk_end': final_frame,
        #         'label': 'no_behavior',
        #         'agent_id': None,
        #         'target_id': None,
        #         'first_chunk': False,
        #         'last_chunk': True,
        #         'angle': 0,
        #     })

    return chunks

In [11]:
def split_files_into_train_test(path, train_df_path):
    all_files = []
    train_df = pd.read_csv(train_df_path)
    final_files = []

    # Gather all file paths
    for dirpath, _, filenames in os.walk(path):
        for filename in filenames:
            all_files.append(os.path.join(dirpath, filename))

    for _, row in train_df.iterrows():
        video_path = os.path.join(path, 'train_annotation', str(row['lab_id']), f"{row['video_id']}.parquet")
        if video_path not in all_files:
            continue
        else:
            final_files.append(video_path)


    train_files, test_files = train_test_split(final_files, test_size=0.1, random_state=42)

    return train_files, test_files

In [12]:
solo_encoder.classes_

array(['no_behavior', 'rear', 'selfgroom', 'dig', 'rest', 'climb',
       'object', 'huddle'], dtype='<U11')

In [13]:
class MABEModelDataset(Dataset):
    def __init__(self, path, solo_encoder, dual_encoder, chunks):
        self.agent_encoder = agent_encoder
        self.path = path
        self.solo_encoder = solo_encoder
        self.dual_encoder = dual_encoder
        self.chunk_size = CHUNK_SIZE
        self.sub_seq_length = SUB_SEQ_LENGTH
        self.sliding_window = SLIDING_WINDOW
        self.all_files = []

        self.chunks = chunks 
        print(f"num chunks: {len(self.chunks)}")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        try:
            info = self.chunks[idx]
            video_id = info['video_id']
            lab_id = info['lab_id']
            width = info['width']
            height = info['height']
            chunk_start = info['chunk_start']
            chunk_end = info['chunk_end']
            is_start = info['first_chunk']
            is_end = info['last_chunk']
            angle = info['angle']
            # print(f"for video {video_id} {lab_id} {chunk_start} {chunk_end}")
        
            half_window = self.sub_seq_length // 2  # 10 for length 21
        
            # Load annotations once to know full video length
            ann_df = pd.read_parquet(os.path.join(self.path, 'train_annotation', lab_id, f"{video_id}.parquet"))
            total_video_length = ann_df['stop_frame'].max() + 1
        
            # --- Adjust chunk boundaries for context frames ---
            load_start = chunk_start
            load_end = chunk_end
            if not is_start:
                load_start = max(0, chunk_start - half_window)
            if not is_end:
                load_end = min(total_video_length - 1, chunk_end + half_window)
        
            # --- Load frames with context ---
            df = load_and_process_video(video_id, lab_id, load_start, load_end, width, height, angle=angle)
            vec_seq = df.to_numpy()
            vec_seq = np.where(np.isnan(vec_seq), -1, vec_seq)
        
            # --- Pad edges only if we’re at start or end ---
            if is_start:
                left_pad = np.repeat(vec_seq[0:1], half_window, axis=0)
                vec_seq = np.concatenate([left_pad, vec_seq], axis=0)
        
            if is_end:
                right_pad = np.repeat(vec_seq[-1:], half_window, axis=0)
                vec_seq = np.concatenate([vec_seq, right_pad], axis=0)
        
            # --- Recompute true valid range within vec_seq ---
            start_offset = 0 if is_start else half_window
            end_offset = vec_seq.shape[0] - half_window if is_end else vec_seq.shape[0] - half_window
        
            # --- Build centered subsequences (length = 21) ---
            sub_seqs = np.stack([
                vec_seq[i - half_window:i + half_window + 1]
                for i in range(half_window, vec_seq.shape[0] - half_window)
            ])  # shape: [num_frames_in_chunk, 21, state_dim]
        
            original_len = sub_seqs.shape[0]
        
            # --- Build frame-level labels for the original (unpadded) chunk ---
            frame_labels = {i: np.full(original_len, 'no_behavior', dtype=object) for i in range(1, 5)}
            frame_targets = {i: np.full(original_len, 0, dtype=int) for i in range(1, 5)}
        
            # Filter annotations that overlap with current chunk
            for _, row in ann_df.iterrows():
                start, stop, action, agent, target = row['start_frame'], row['stop_frame'], remapping_dict[row['action']], row['agent_id'], row['target_id']
        
                # Skip if outside current chunk boundaries
                if stop < chunk_start or start > chunk_end or action == "no_behavior":
                    continue
        
                # Convert to local frame indices within current chunk
                local_start = max(0, start - chunk_start)
                local_stop = min(original_len - 1, stop - chunk_start)
        
                frame_labels[agent][local_start:local_stop + 1] = action
                frame_targets[agent][local_start:local_stop + 1] = target
        
            # --- Extract labels for the center frame (index 10) of each subsequence ---
            mouse_labels = {i: [] for i in range(1, 5)}
            mouse_targets = {i: [] for i in range(1, 5)}
        
            for i in range(original_len):
                center_frame = i + chunk_start  # actual frame index in video
                for mouse_id in range(1, 5):
                    label = frame_labels[mouse_id][i]
                    target = frame_targets[mouse_id][i]
                    mouse_labels[mouse_id].append(label)
                    mouse_targets[mouse_id].append(target)
        
            num_agents = 4
            num_solo_classes = len(self.solo_encoder.classes_)
            num_dual_classes = len(self.dual_encoder.classes_)

            # Shapes:
            # solo_labels: [T, 4]  (long ints: class indices for solo encoder)
            # dual_labels: [T, 4, 4] (long ints: class indices for dual encoder; diagonal stays 0)
            solo_labels = torch.zeros((original_len, num_agents), dtype=torch.long)
            dual_labels = torch.zeros((original_len, num_agents, num_agents), dtype=torch.long)

            for f in range(original_len):
                for agent in range(1, num_agents + 1):
                    action_str = frame_labels[agent][f]  # e.g. 'attack' or 'no_behavior'

                    # --- SOLO: encode in separate solo_labels matrix ---
                    if action_str in self_actions:
                        solo_idx = self.solo_encoder.transform([action_str])[0]
                        solo_labels[f, agent - 1] = solo_idx

                    # --- DUAL: encode only non-self, non-zero targets ---
                    target = frame_targets[agent][f]  # 0 means no target
                    if target != 0 and target != agent:
                        target_idx = self.agent_encoder.transform([target])[0]  # e.g. 0..3
                        dual_idx = self.dual_encoder.transform([action_str])[0]
                        dual_labels[f, agent - 1, target_idx] = dual_idx
            states = sub_seqs
        
            # --- Sanity check ---
            # print(f"Chunk [{chunk_start}, {chunk_end}] | vec_seq: {vec_seq.shape} | subseqs: {sub_seqs.shape}")
        
            # for i in range(min(20, solo_labels.shape[0])):  # print first few
            #     # for j in range(1, 5):
            #     j = 2
            #     print(f"Frame {i + chunk_start} | dual labels: {dual_labels[i]}")
            # print(f"states shape {states.shape}")

            # mouse_states = {}
            # for i in range(4):
            #     start_col = i * 19
            #     end_col = (i + 1) * 19
            #     mouse_states[i + 1] = states[:,:, start_col:end_col] 
            # dist_feats = states[:, :, 76:]

            has_action = torch.zeros((original_len, 1), dtype=torch.long)

            for f in range(original_len):
                solo_nonzero = (solo_labels[f] != 0).any()  # if any agent has a solo action
                dual_nonzero = (dual_labels[f] != 0).any()  # if any agent-target pair has a dual action
                if solo_nonzero or dual_nonzero:
                    has_action[f, 0] = 1
        
            return (
                # mouse_states[1],
                # mouse_states[2],
                # mouse_states[3],
                # mouse_states[4],
                # dist_feats,
                states,
                solo_labels,
                dual_labels,
                has_action,
                # video_id,
                # chunk_start
            )
        except:
            print("returning None")
            return None
    
class MABEModelDatasetEval(Dataset):
    def __init__(self, path, solo_encoder, dual_encoder, chunks):
        self.agent_encoder = agent_encoder
        self.path = path
        self.solo_encoder = solo_encoder
        self.dual_encoder = dual_encoder
        self.chunk_size = CHUNK_SIZE
        self.sub_seq_length = SUB_SEQ_LENGTH
        self.sliding_window = SLIDING_WINDOW
        self.all_files = []

        self.chunks = chunks 
        print(f"num chunks: {len(self.chunks)}")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        # try:
        info = self.chunks[idx]
        video_id = info['video_id']
        lab_id = info['lab_id']
        width = info['width']
        height = info['height']
        chunk_start = info['chunk_start']
        chunk_end = info['chunk_end']
        is_start = info['first_chunk']
        is_end = info['last_chunk']
        angle = info['angle']
        # print(f"for video {video_id} {lab_id} {chunk_start} {chunk_end}")
    
        half_window = self.sub_seq_length // 2  # 10 for length 21
    
        # Load annotations once to know full video length
        ann_df = pd.read_parquet(os.path.join(self.path, 'train_annotation', lab_id, f"{video_id}.parquet"))
        total_video_length = ann_df['stop_frame'].max() + 1
    
        # --- Adjust chunk boundaries for context frames ---
        load_start = chunk_start
        load_end = chunk_end
        if not is_start:
            load_start = max(0, chunk_start - half_window)
        if not is_end:
            load_end = min(total_video_length - 1, chunk_end + half_window)
    
        # --- Load frames with context ---
        df = load_and_process_video(video_id, lab_id, load_start, load_end, width, height, angle=angle)
        vec_seq = df.to_numpy()
        vec_seq = np.where(np.isnan(vec_seq), -1, vec_seq)
    
        # --- Pad edges only if we’re at start or end ---
        if is_start:
            left_pad = np.repeat(vec_seq[0:1], half_window, axis=0)
            vec_seq = np.concatenate([left_pad, vec_seq], axis=0)
    
        if is_end:
            right_pad = np.repeat(vec_seq[-1:], half_window, axis=0)
            vec_seq = np.concatenate([vec_seq, right_pad], axis=0)
    
        # --- Recompute true valid range within vec_seq ---
        start_offset = 0 if is_start else half_window
        end_offset = vec_seq.shape[0] - half_window if is_end else vec_seq.shape[0] - half_window
    
        # --- Build centered subsequences (length = 21) ---
        sub_seqs = np.stack([
            vec_seq[i - half_window:i + half_window + 1]
            for i in range(half_window, vec_seq.shape[0] - half_window)
        ])  # shape: [num_frames_in_chunk, 21, state_dim]
    
        original_len = sub_seqs.shape[0]
    
        # --- Build frame-level labels for the original (unpadded) chunk ---
        frame_labels = {i: np.full(original_len, 'no_behavior', dtype=object) for i in range(1, 5)}
        frame_targets = {i: np.full(original_len, 0, dtype=int) for i in range(1, 5)}
    
        # Filter annotations that overlap with current chunk
        for _, row in ann_df.iterrows():
            start, stop, action, agent, target = row['start_frame'], row['stop_frame'], remapping_dict[row['action']], row['agent_id'], row['target_id']
    
            # Skip if outside current chunk boundaries
            if stop < chunk_start or start > chunk_end or action == "no_behavior":
                continue
    
            # Convert to local frame indices within current chunk
            local_start = max(0, start - chunk_start)
            local_stop = min(original_len - 1, stop - chunk_start)
    
            frame_labels[agent][local_start:local_stop + 1] = action
            frame_targets[agent][local_start:local_stop + 1] = target
    
        # --- Extract labels for the center frame (index 10) of each subsequence ---
        mouse_labels = {i: [] for i in range(1, 5)}
        mouse_targets = {i: [] for i in range(1, 5)}
    
        for i in range(original_len):
            center_frame = i + chunk_start  # actual frame index in video
            for mouse_id in range(1, 5):
                label = frame_labels[mouse_id][i]
                target = frame_targets[mouse_id][i]
                mouse_labels[mouse_id].append(label)
                mouse_targets[mouse_id].append(target)
    
        num_agents = 4
        num_solo_classes = len(self.solo_encoder.classes_)
        num_dual_classes = len(self.dual_encoder.classes_)

        # Shapes:
        # solo_labels: [T, 4]  (long ints: class indices for solo encoder)
        # dual_labels: [T, 4, 4] (long ints: class indices for dual encoder; diagonal stays 0)
        solo_labels = torch.zeros((original_len, num_agents), dtype=torch.long)
        dual_labels = torch.zeros((original_len, num_agents, num_agents), dtype=torch.long)

        for f in range(original_len):
            for agent in range(1, num_agents + 1):
                action_str = frame_labels[agent][f]  # e.g. 'attack' or 'no_behavior'

                # --- SOLO: encode in separate solo_labels matrix ---
                if action_str in self_actions:
                    solo_idx = self.solo_encoder.transform([action_str])[0]
                    solo_labels[f, agent - 1] = solo_idx

                # --- DUAL: encode only non-self, non-zero targets ---
                target = frame_targets[agent][f]  # 0 means no target
                if target != 0 and target != agent:
                    target_idx = self.agent_encoder.transform([target])[0]  # e.g. 0..3
                    dual_idx = self.dual_encoder.transform([action_str])[0]
                    dual_labels[f, agent - 1, target_idx] = dual_idx
        states = sub_seqs
    
        # --- Sanity check ---
        # print(f"Chunk [{chunk_start}, {chunk_end}] | vec_seq: {vec_seq.shape} | subseqs: {sub_seqs.shape}")
    
        # for i in range(min(20, solo_labels.shape[0])):  # print first few
        #     # for j in range(1, 5):
        #     j = 2
        #     print(f"Frame {i + chunk_start} | dual labels: {dual_labels[i]}")
        # print(f"states shape {states.shape}")

        # mouse_states = {}
        # for i in range(4):
        #     start_col = i * 19
        #     end_col = (i + 1) * 19
        #     mouse_states[i + 1] = states[:,:, start_col:end_col] 
        # dist_feats = states[:, :, 76:]

        has_action = torch.zeros((original_len, 1), dtype=torch.long)

        for f in range(original_len):
            solo_nonzero = (solo_labels[f] != 0).any()  # if any agent has a solo action
            dual_nonzero = (dual_labels[f] != 0).any()  # if any agent-target pair has a dual action
            if solo_nonzero or dual_nonzero:
                has_action[f, 0] = 1
    
        return (
            # mouse_states[1],
            # mouse_states[2],
            # mouse_states[3],
            # mouse_states[4],
            # dist_feats,
            states,
            solo_labels,
            dual_labels,
            has_action,
            video_id,
            chunk_start
        )
        # except:
        #     print("returning None")
        #     return None

In [14]:
def filter_chunks_by_inactive_frames(chunks, inactive_threshold=CHUNK_SIZE):
    filtered_chunks = []

    for info in chunks:
        lab_id = info['lab_id']
        video_id = info['video_id']
        chunk_start = info['chunk_start']
        chunk_end = info['chunk_end']

        annotation_path = os.path.join(DATA_PATH, 'train_annotation', lab_id, f"{video_id}.parquet")
        if not os.path.exists(annotation_path):
            print(f"Warning: Annotation file not found for {video_id} in {lab_id}")
            continue

        df_annotations = pd.read_parquet(annotation_path)

        # Frames in chunk
        chunk_frames = set(range(chunk_start, chunk_end + 1))

        # Filter annotations that overlap with chunk
        overlapping = df_annotations[
            (df_annotations['stop_frame'] >= chunk_start) &
            (df_annotations['start_frame'] <= chunk_end)
        ]

        # Collect active frames
        active_frames = set()
        for _, row in overlapping.iterrows():
            start = max(row['start_frame'], chunk_start)
            stop = min(row['stop_frame'], chunk_end)
            active_frames.update(range(start, stop + 1))

        # Determine inactive frames
        inactive_frames = chunk_frames - active_frames

        if len(inactive_frames) <= inactive_threshold:
            info['inactive_frames'] = len(inactive_frames)
            filtered_chunks.append(info)

    return filtered_chunks

# chunks = filter_chunks_by_inactive_frames(chunks, inactive_threshold=100)
# print(len(chunks))
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)


In [15]:
# train_files, test_files = split_files_into_train_test(DATA_PATH, os.path.join(DATA_PATH, 'train.csv'))
# train_chunks = build_chunks(DATA_PATH, os.path.join(DATA_PATH, "train.csv"), train_files[:5])
# train_chunks = filter_chunks_by_inactive_frames(train_chunks, 100)
# chunk = train_chunks[10]
# new_chunk = chunk.copy()
# new_chunk['angle'] = 180
# new_chunks = [chunk, new_chunk]
# dataset = MABEModelDataset(DATA_PATH, solo_encoder, dual_encoder, new_chunks)
# i = 1
# all_batches = []
# for batch in dataset:
#     a, b, c, d, e, f, g, h = batch
#     all_batches.append(batch)
#     # if i % 1 == 0:
#     #     break
#     # else:
#     #     i += 1
# print(a.shape)
# print(d.shape)
# print(e.shape)
# print(h.shape)

In [16]:
# all_batches[0][6]

In [17]:
class ResidualConv1DBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, causal=False):
        super().__init__()
        self.causal = causal
        padding = (kernel_size - 1) if causal else (kernel_size - 1) // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size,
                               padding=padding, padding_mode='replicate')
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size,
                               padding=padding, padding_mode='replicate')
        self.bn2 = nn.BatchNorm1d(channels)
        self.activation = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.activation(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        # If causal, cut future timesteps (preserve only past+present)
        if self.causal:
            # ✅ Keep same seq_len as input
            cut = out.shape[-1] - identity.shape[-1]
            if cut > 0:
                out = out[:, :, :-cut]

        out += identity[..., :out.shape[-1]]
        return self.activation(out)




# ============================================================
# 2️⃣ Fully Connected Residual Block
# ============================================================
class ResidualFCBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.act(self.bn1(self.fc1(x)))
        out = self.bn2(self.fc2(out))
        out += identity
        return self.act(out)




# ============================================================
# 3️⃣ Main BehaviorNet (MABe-style)
# ============================================================
class BehaviorNet(nn.Module):
    def __init__(
        self,
        input_dim=76,
        seq_len=21,
        embed_channels=64,
        contexter_channels=64,
        embed_blocks=8,     # tuned from paper (8 residual blocks)
        context_blocks=8,
        kernel_size=3,
        behavior_classes=38,
        agent_classes=5,
        num_mice=4,
        annotator_embedding_dim=None,
        dropout=0.0,
        num_solo_classes = 8,
        num_dual_classes = 13
    ):
        super().__init__()
        self.num_mice = num_mice
        self.seq_len = seq_len

        # ---------- Embedder (non-causal residual convs) ----------
        self.embed_in = nn.Conv1d(input_dim, embed_channels, 1)
        self.embed_blocks = nn.Sequential(*[
            ResidualConv1DBlock(embed_channels, kernel_size, causal=False)
            for _ in range(embed_blocks)
        ])

        # ---------- Contexter (causal residual convs) ----------
        self.context_in = nn.Conv1d(embed_channels, contexter_channels, 1)
        self.context_blocks = nn.Sequential(*[
            ResidualConv1DBlock(contexter_channels, kernel_size, causal=True)
            for _ in range(context_blocks)
        ])

        # ---------- Shared FC block ----------
        fc_input_dim = contexter_channels
        if annotator_embedding_dim:
            fc_input_dim += annotator_embedding_dim

        self.shared_fc = ResidualFCBlock(fc_input_dim)

        # --- SOLO Action Head ---
        self.solo_mlp = nn.Sequential(
            nn.Linear(fc_input_dim, num_solo_classes * self.num_mice)
        )

        # --- DUAL Action Head ---
        self.dual_mlp = nn.Sequential(
            nn.Linear(fc_input_dim, num_dual_classes * self.num_mice * self.num_mice)
        )

        self.dropout = nn.Dropout(dropout)
        self.num_solo_classes = num_solo_classes
        self.num_dual_classes = num_dual_classes

    def forward(self, x, annotator_embed=None):
        """
        x: (B, seq_len, input_dim)
        annotator_embed: optional (B, annotator_embedding_dim)
        """
        # x = x.squeeze(0)  # remove the batch dim → (25000, 21, 32)
        B = x.shape[0]

        x = x.permute(0, 2, 1)  # -> (B, input_dim, seq_len)
        x = self.embed_in(x)
        x = self.embed_blocks(x)

        x = self.context_in(x)
        x = self.context_blocks(x)  # (B, contexter_channels, seq_len)

        # take center frame (causal ensures current frame summary)
        center_idx = self.seq_len // 2
        x = x[:, :, center_idx]  # (B, contexter_channels)

        if annotator_embed is not None:
            x = torch.cat([x, annotator_embed], dim=-1)

        x = self.shared_fc(x)
        x = self.dropout(x)

        solo_logits = self.solo_mlp(x)
        dual_logits = self.dual_mlp(x)
        solo_logits = solo_logits.view(B, self.num_mice, self.num_solo_classes)
        dual_logits = dual_logits.view(B, self.num_mice, self.num_mice, self.num_dual_classes)

        return solo_logits, dual_logits


In [18]:
from sklearn.utils import resample
from collections import defaultdict
import random
import copy

def stratified_resample(chunks, min_samples_per_class, max_samples_per_class):
    label_to_chunks = defaultdict(list)

    # Group chunks by label
    for chunk in chunks:
        label_to_chunks[chunk['label']].append(chunk)

    print("Original class distribution:")
    for label, samples in label_to_chunks.items():
        print(f"  Class '{label}': {len(samples)} chunks")

    resampled_chunks = []
    new_label_to_chunks = defaultdict(list)

    for label, samples in label_to_chunks.items():
        n_samples = len(samples)

        # Special handling for "no_behavior"
        # effective_max = 10000 if label == "no_behavior" else max_samples_per_class
        effective_max = max_samples_per_class

        # --- CASE 1: Underrepresented class ---
        if n_samples < min_samples_per_class:
            augmented = []

            # Step 1: Generate rotated variants for each chunk
            rotation_angles = [90, 180, 270]
            for chunk in samples:
                for angle in rotation_angles:
                    new_chunk = copy.deepcopy(chunk)
                    new_chunk["angle"] = angle
                    augmented.append(new_chunk)

            # Combine originals + augmented
            all_possible = samples + augmented

            # Step 2: If we have enough unique variants, pick min_samples_per_class
            if len(all_possible) >= min_samples_per_class:
                resampled = random.sample(all_possible, min_samples_per_class)
            else:
                # Step 3: If still short, oversample from this pool
                resampled = resample(all_possible,
                                     replace=True,
                                     n_samples=min_samples_per_class,
                                     random_state=42)

        # --- CASE 2: Overrepresented class ---
        elif n_samples > effective_max:
            resampled = random.sample(samples, effective_max)

        # --- CASE 3: Just right ---
        else:
            resampled = samples

        resampled_chunks.extend(resampled)
        new_label_to_chunks[label] = resampled

    print("\nPost-resampling class distribution:")
    for label, samples in new_label_to_chunks.items():
        print(f"  Class '{label}': {len(samples)} chunks")

    random.shuffle(resampled_chunks)
    return resampled_chunks


In [19]:
def collate_fn(batch):
    # filter out Nones
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return torch.utils.data.default_collate(batch)

In [20]:
train_files, test_files = split_files_into_train_test(DATA_PATH, os.path.join(DATA_PATH, 'train.csv'))
train_chunks = build_chunks(DATA_PATH, os.path.join(DATA_PATH, "train.csv"), train_files)
train_chunks = filter_chunks_by_inactive_frames(train_chunks, 100)

train_chunks = stratified_resample(train_chunks, min_samples_per_class=1250, max_samples_per_class=2500)
# test_chunks = build_chunks_test(DATA_PATH, os.path.join(DATA_PATH, "train.csv"), test_files)
test_chunks = build_chunks(DATA_PATH, os.path.join(DATA_PATH, "train.csv"), test_files)

Building chunks with 762 files
Original class distribution:
  Class 'rear': 3410 chunks
  Class 'avoid': 500 chunks
  Class 'attack': 4628 chunks
  Class 'approach': 1588 chunks
  Class 'chase': 514 chunks
  Class 'shepherd': 153 chunks
  Class 'sniff': 27086 chunks
  Class 'mount': 2569 chunks
  Class 'disengage': 214 chunks
  Class 'selfgroom': 875 chunks
  Class 'escape': 1061 chunks
  Class 'dominance': 294 chunks
  Class 'defend': 779 chunks
  Class 'dig': 929 chunks
  Class 'rest': 232 chunks
  Class 'climb': 868 chunks
  Class 'follow': 153 chunks
  Class 'object': 89 chunks
  Class 'huddle': 183 chunks

Post-resampling class distribution:
  Class 'rear': 2500 chunks
  Class 'avoid': 1250 chunks
  Class 'attack': 2500 chunks
  Class 'approach': 1588 chunks
  Class 'chase': 1250 chunks
  Class 'shepherd': 1250 chunks
  Class 'sniff': 2500 chunks
  Class 'mount': 2500 chunks
  Class 'disengage': 1250 chunks
  Class 'selfgroom': 1250 chunks
  Class 'escape': 1250 chunks
  Class 'do

In [21]:
# train_chunks, test_chunks = train_chunks[:50], test_chunks[:50]

In [22]:
train_dataset = MABEModelDataset(DATA_PATH, solo_encoder, dual_encoder, train_chunks)
test_dataset = MABEModelDatasetEval(DATA_PATH, solo_encoder, dual_encoder, test_chunks)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

num chunks: 29088
num chunks: 4175


In [23]:
model = BehaviorNet()
model.load_state_dict(torch.load(r"C:\Coding\mabe\MABe_2022_TVAE\umair_lesser_epoch5.pt"))
model = model.to("cuda")
D = 64
N = 4  # number of agents
# dataloader = DataLoader(
#     dataset,                # Your dataset
#     batch_size=1,     # Set your batch size
#     shuffle=True,      # Shuffle data at every epoch,
#     num_workers=4,
#     # collate_fn=truncate_collate_fn
# )
weights = torch.ones(len(solo_encoder.classes_))
weights = weights.to("cuda")
weights[0] = 0.01
weights2 = torch.ones(len(dual_encoder.classes_))
weights2 = weights2.to("cuda")
weights2[0] = 0.01
pos_weight = torch.tensor([0.1]).to("cuda")

solo_loss_fn = nn.CrossEntropyLoss(weight=weights)
dual_loss_fn = nn.CrossEntropyLoss(weight=weights2)
has_action_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 100

C:\Users\PC\AppData\Local\Temp\ipykernel_17588\2623601735.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(r"C:\Coding\mabe\MABe_2022_TVA

In [24]:
# chunks = build_chunks(DATA_PATH, os.path.join(DATA_PATH, "train.csv"))
# chunks = filter_chunks_by_inactive_frames(chunks, 100)

# train_chunks, test_chunks = train_test_split(chunks, test_size=0.01, random_state=42)
# train_chunks = stratified_resample(train_chunks, min_samples_per_class=1250, max_samples_per_class=2500)
# # train_chunks, test_chunks = train_chunks[:50], test_chunks[:50]
# train_dataset = MABEModelDataset(DATA_PATH, solo_encoder, dual_encoder, train_chunks)
# test_dataset = MABEModelDataset(DATA_PATH, solo_encoder, dual_encoder, test_chunks)

# train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
# model = AgentActionModel()
# model.load_state_dict(torch.load(r"C:\Coding\mabe\MABe_2022_TVAE\masked_cnn_attent_epoch7.pt"))
# model = model.to("cuda")
# D = 64
# N = 4  # number of agents
# # dataloader = DataLoader(
# #     dataset,                # Your dataset
# #     batch_size=1,     # Set your batch size
# #     shuffle=True,      # Shuffle data at every epoch,
# #     num_workers=4,
# #     # collate_fn=truncate_collate_fn
# # )
# weights = torch.ones(len(solo_encoder.classes_))
# weights = weights.to("cuda")
# weights[0] = 0.09
# weights2 = torch.ones(len(dual_encoder.classes_))
# weights2 = weights2.to("cuda")
# weights2[0] = 0.09

# solo_loss_fn = nn.CrossEntropyLoss(weight=weights)
# dual_loss_fn = nn.CrossEntropyLoss(weight=weights2)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# epochs = 100

In [25]:
# def extract_behavior_segments(solo_labels, dual_labels, confidence, chunk_start, video_id, test_df, min_conf_threshold=0.5):
#     results = []

#     # Get the list of labeled behaviors and lab_id for this video
#     row = test_df.loc[test_df['video_id'] == video_id]
#     values = ast.literal_eval(row['behaviors_labeled'].values[0])
#     lab_id = row['lab_id'].values[0]

#     num_frames = solo_labels.shape[0]
#     num_mice = solo_labels.shape[1]
#     chunk_start = int(chunk_start)

#     # Create a boolean mask for confident frames
#     conf_mask = (confidence.squeeze() > min_conf_threshold)
#     total_frames = num_frames
#     dropped_frames = (~conf_mask).sum().item()


#     # ---- SOLO ACTIONS ----
#     for i in range(num_mice):
#         prev_label = 'no_behavior'
#         start = None
#         for f in range(num_frames):
#             # skip frames below confidence threshold
#             if not conf_mask[f]:
#                 curr_label = 'no_behavior'
#             else:
#                 curr_label = solo_labels[f, i]

#             if curr_label != prev_label:
#                 if prev_label != 'no_behavior':
#                     entry = {
#                         'lab_id': lab_id,
#                         'video_id': video_id,
#                         'agent_id': f'mouse{i+1}',
#                         'target_id': 'self',
#                         'action': prev_label,
#                         'behaviors_labeled': f'["mouse{i+1},self,{prev_label}"]',
#                         'start_frame': chunk_start + start,
#                         'stop_frame': chunk_start + f - 1,
#                     }
#                     entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#                     if entry_str in values:
#                         results.append(entry)
#                 if curr_label != 'no_behavior':
#                     start = f
#                 prev_label = curr_label

#         # Handle last ongoing solo behavior
#         if prev_label != 'no_behavior':
#             entry = {
#                 'lab_id': lab_id,
#                 'video_id': video_id,
#                 'agent_id': f'mouse{i+1}',
#                 'target_id': 'self',
#                 'action': prev_label,
#                 'behaviors_labeled': f'["mouse{i+1},self,{prev_label}"]',
#                 'start_frame': chunk_start + start,
#                 'stop_frame': chunk_start + num_frames - 1,
#             }
#             entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#             if entry_str in values:
#                 results.append(entry)

#     # ---- DUAL ACTIONS ----
#     for i in range(num_mice):
#         for j in range(num_mice):
#             if i == j:
#                 continue
#             prev_label = 'no_behavior'
#             start = None
#             for f in range(num_frames):
#                 # skip frames below confidence threshold
#                 if not conf_mask[f]:
#                     curr_label = 'no_behavior'
#                 else:
#                     curr_label = dual_labels[f, i, j]

#                 if curr_label != prev_label:
#                     if prev_label != 'no_behavior':
#                         entry = {
#                             'lab_id': lab_id,
#                             'video_id': video_id,
#                             'agent_id': f'mouse{i+1}',
#                             'target_id': f'mouse{j+1}',
#                             'action': prev_label,
#                             'behaviors_labeled': f'["mouse{i+1},mouse{j+1},{prev_label}"]',
#                             'start_frame': chunk_start + start,
#                             'stop_frame': chunk_start + f - 1,
#                         }
#                         entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#                         if entry_str in values:
#                             results.append(entry)
#                     if curr_label != 'no_behavior':
#                         start = f
#                     prev_label = curr_label

#             # Handle last ongoing dual behavior
#             if prev_label != 'no_behavior':
#                 entry = {
#                     'lab_id': lab_id,
#                     'video_id': video_id,
#                     'agent_id': f'mouse{i+1}',
#                     'target_id': f'mouse{j+1}',
#                     'action': prev_label,
#                     'behaviors_labeled': f'["mouse{i+1},mouse{j+1},{prev_label}"]',
#                     'start_frame': chunk_start + start,
#                     'stop_frame': chunk_start + num_frames - 1,
#                 }
#                 entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#                 if entry_str in values:
#                     results.append(entry)

#     return pd.DataFrame(results), dropped_frames, total_frames


def extract_behavior_segments(solo_labels, dual_labels, chunk_start, video_id, test_df):
    results = []

    # Get the list of labeled behaviors and lab_id for this video
    row = test_df.loc[test_df['video_id'] == video_id]
    values1 = ast.literal_eval(row['behaviors_labeled'].values[0])
    values = []

    for item in values1:
        p1, p2, action = item.split(",", 2)   # split into 3 parts
        new_action = remapping_dict.get(action, action)
        # print(f"{p1},{p2},{new_action}")
        values.append(f"{p1},{p2},{new_action}")

    lab_id = row['lab_id'].values[0]

    num_frames = solo_labels.shape[0]
    num_mice = solo_labels.shape[1]
    chunk_start = int(chunk_start)


    # ---- SOLO ACTIONS ----
    for i in range(num_mice):
        prev_label = 'no_behavior'
        start = None
        for f in range(num_frames):
            # skip frames below confidence threshold
            curr_label = solo_labels[f, i]

            if curr_label != prev_label:
                if prev_label != 'no_behavior':
                    entry = {
                        'lab_id': lab_id,
                        'video_id': video_id,
                        'agent_id': f'mouse{i+1}',
                        'target_id': 'self',
                        'action': prev_label,
                        'behaviors_labeled': f'["mouse{i+1},self,{prev_label}"]',
                        'start_frame': chunk_start + start,
                        'stop_frame': chunk_start + f - 1,
                    }
                    entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
                    if entry_str in values:
                        results.append(entry)
                if curr_label != 'no_behavior':
                    start = f
                prev_label = curr_label

        # Handle last ongoing solo behavior
        if prev_label != 'no_behavior':
            entry = {
                'lab_id': lab_id,
                'video_id': video_id,
                'agent_id': f'mouse{i+1}',
                'target_id': 'self',
                'action': prev_label,
                'behaviors_labeled': f'["mouse{i+1},self,{prev_label}"]',
                'start_frame': chunk_start + start,
                'stop_frame': chunk_start + num_frames - 1,
            }
            entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
            if entry_str in values:
                results.append(entry)

    # ---- DUAL ACTIONS ----
    for i in range(num_mice):
        for j in range(num_mice):
            if i == j:
                continue
            prev_label = 'no_behavior'
            start = None
            for f in range(num_frames):
                curr_label = dual_labels[f, i, j]

                if curr_label != prev_label:
                    if prev_label != 'no_behavior':
                        entry = {
                            'lab_id': lab_id,
                            'video_id': video_id,
                            'agent_id': f'mouse{i+1}',
                            'target_id': f'mouse{j+1}',
                            'action': prev_label,
                            'behaviors_labeled': f'["mouse{i+1},mouse{j+1},{prev_label}"]',
                            'start_frame': chunk_start + start,
                            'stop_frame': chunk_start + f - 1,
                        }
                        entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
                        if entry_str in values:
                            results.append(entry)
                    if curr_label != 'no_behavior':
                        start = f
                    prev_label = curr_label

            # Handle last ongoing dual behavior
            if prev_label != 'no_behavior':
                entry = {
                    'lab_id': lab_id,
                    'video_id': video_id,
                    'agent_id': f'mouse{i+1}',
                    'target_id': f'mouse{j+1}',
                    'action': prev_label,
                    'behaviors_labeled': f'["mouse{i+1},mouse{j+1},{prev_label}"]',
                    'start_frame': chunk_start + start,
                    'stop_frame': chunk_start + num_frames - 1,
                }
                entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
                if entry_str in values:
                    results.append(entry)

    return pd.DataFrame(results)

In [29]:
"""F Beta customized for the data format of the MABe challenge — now extended to show precision, recall, accuracy, etc."""

import json
from collections import defaultdict
import pandas as pd
import polars as pl


class HostVisibleError(Exception):
    pass


def compute_metrics(tps, fps, fns, beta):
    """Return precision, recall, fbeta, accuracy."""
    tp = tps
    fp = fps
    fn = fns

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    if precision == 0 and recall == 0:
        fbeta = 0.0
    else:
        fbeta = (1 + beta**2) * precision * recall / (beta**2 * precision + recall)

    # For these tasks: accuracy = tp / (tp + fp + fn)
    denom = tp + fp + fn
    accuracy = tp / denom if denom > 0 else 0.0

    return precision, recall, fbeta, accuracy


def single_lab_f1(lab_solution: pl.DataFrame, lab_submission: pl.DataFrame, beta: float = 1, verbose=False):
    label_frames: defaultdict[str, set[int]] = defaultdict(set)
    prediction_frames: defaultdict[str, set[int]] = defaultdict(set)

    for row in lab_solution.to_dicts():
        label_frames[row['label_key']].update(range(row['start_frame'], row['stop_frame']))

    for video in lab_solution['video_id'].unique():
        active_labels: str = lab_solution.filter(pl.col('video_id') == video)['behaviors_labeled'].first()
        active_labels: set[str] = set(json.loads(active_labels))
        predicted_mouse_pairs: defaultdict[str, set[int]] = defaultdict(set)

        for row in lab_submission.filter(pl.col('video_id') == video).to_dicts():
            if ','.join([str(row['agent_id']), str(row['target_id']), row['action']]) not in active_labels:
                continue

            new_frames = set(range(row['start_frame'], row['stop_frame']))
            new_frames = new_frames.difference(prediction_frames[row['prediction_key']])
            prediction_pair = ','.join([str(row['agent_id']), str(row['target_id'])])

            if predicted_mouse_pairs[prediction_pair].intersection(new_frames):
                raise HostVisibleError('Multiple predictions for same frame from one agent/target pair')

            prediction_frames[row['prediction_key']].update(new_frames)
            predicted_mouse_pairs[prediction_pair].update(new_frames)

    tps = defaultdict(int)
    fns = defaultdict(int)
    fps = defaultdict(int)

    # Compare predictions vs labels
    for key, pred_frames in prediction_frames.items():
        action = key.split('_')[-1]
        matched_label_frames = label_frames[key]

        tps[action] += len(pred_frames.intersection(matched_label_frames))
        fns[action] += len(matched_label_frames.difference(pred_frames))
        fps[action] += len(pred_frames.difference(matched_label_frames))

    # Also handle keys with no predictions
    distinct_actions = set()
    for key, frames in label_frames.items():
        action = key.split('_')[-1]
        distinct_actions.add(action)
        if key not in prediction_frames:
            fns[action] += len(frames)

    # Compute per-action metrics
    action_fbeta = []
    action_metrics = {}

    for action in distinct_actions:
        p, r, f, acc = compute_metrics(tps[action], fps[action], fns[action], beta)
        action_fbeta.append(f)
        action_metrics[action] = {
            "precision": p,
            "recall": r,
            "fbeta": f,
            "accuracy": acc,
            "tp": tps[action],
            "fp": fps[action],
            "fn": fns[action],
        }

    if verbose:
        # Aggregate TP/FP/FN across all actions for this lab
        lab_tp = sum(m["tp"] for m in action_metrics.values())
        lab_fp = sum(m["fp"] for m in action_metrics.values())
        lab_fn = sum(m["fn"] for m in action_metrics.values())

        # Compute combined metrics
        lab_precision, lab_recall, lab_fbeta, lab_accuracy = compute_metrics(
            lab_tp, lab_fp, lab_fn, beta
        )

        print(f"\n=== Metrics For This Lab {lab_solution['lab_id'][0]} ===")
        print(f"  TP = {lab_tp}")
        print(f"  FP = {lab_fp}")
        print(f"  FN = {lab_fn}")
        print(f"  Precision: {lab_precision:.4f}")
        print(f"  Recall:    {lab_recall:.4f}")
        print(f"  Fβ:        {lab_fbeta:.4f}")
        print(f"  Accuracy:  {lab_accuracy:.4f}")
        print("==============================\n")

    return sum(action_fbeta) / len(action_fbeta), action_metrics


def mouse_fbeta(solution: pd.DataFrame, submission: pd.DataFrame, beta: float = 1, verbose=True) -> float:
    if len(solution) == 0 or len(submission) == 0:
        raise ValueError('Missing solution or submission data')

    expected_cols = ['video_id', 'agent_id', 'target_id', 'action', 'start_frame', 'stop_frame']
    for col in expected_cols:
        if col not in solution.columns:
            raise ValueError(f'Solution is missing column {col}')
        if col not in submission.columns:
            raise ValueError(f'Submission is missing column {col}')

    solution = pl.DataFrame(solution)
    submission = pl.DataFrame(submission)

    assert (solution['start_frame'] <= solution['stop_frame']).all()
    assert (submission['start_frame'] <= submission['stop_frame']).all()

    submission = submission.filter(pl.col('video_id').is_in(set(solution['video_id'].unique())))

    solution = solution.with_columns(
        pl.concat_str(
            [pl.col('video_id'), pl.col('agent_id'), pl.col('target_id'), pl.col('action')],
            separator='_'
        ).alias('label_key')
    )
    submission = submission.with_columns(
        pl.concat_str(
            [pl.col('video_id'), pl.col('agent_id'), pl.col('target_id'), pl.col('action')],
            separator='_'
        ).alias('prediction_key')
    )

    lab_scores = []
    lab_metrics = []

    for lab in solution['lab_id'].unique():
        lab_solution = solution.filter(pl.col('lab_id') == lab).clone()
        lab_submission = submission.filter(pl.col('video_id').is_in(set(lab_solution['video_id'].unique()))).clone()

        score, metrics = single_lab_f1(lab_solution, lab_submission, beta=beta, verbose=verbose)
        lab_scores.append(score)
        lab_metrics.append(metrics)

    # Aggregate across labs
    if verbose:
        print("\n======= GLOBAL SUMMARY =======")
        global_tp = global_fp = global_fn = 0

        for lab_dict in lab_metrics:
            for act, vals in lab_dict.items():
                global_tp += vals["tp"]
                global_fp += vals["fp"]
                global_fn += vals["fn"]

        p, r, f, acc = compute_metrics(global_tp, global_fp, global_fn, beta)

        print(f"Global Precision: {p:.4f}")
        print(f"Global Recall:    {r:.4f}")
        print(f"Global Fβ:        {f:.4f}")
        print(f"Global Accuracy:  {acc:.4f}")
        print("==============================\n")

    return sum(lab_scores) / len(lab_scores)


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str, beta: float = 1, verbose=False) -> float:
    solution = solution.drop(row_id_column_name, axis='columns', errors='ignore')
    submission = submission.drop(row_id_column_name, axis='columns', errors='ignore')
    return mouse_fbeta(solution, submission, beta=beta, verbose=verbose)

In [30]:
def evaluate(model, dataloader, test_df):
    model.eval()
    total_solo_loss = 0
    total_dual_loss = 0
    total_conf_loss = 0
    total_batches = 0
    decoded_dfs = []
    actual_dfs = []
    all_conf = []
    all_has_action = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader)):
            if batch is None:
                continue
            states, solo_labels, dual_labels, has_action, video_id, chunk_start = batch
            states = states[0]
            video_id = int(video_id[0])
            # print(int(video_id))
            chunk_start = chunk_start[0]
            has_action = has_action[0].to("cuda")
            solo_labels, dual_labels = solo_labels[0].to("cuda"), dual_labels[0].to("cuda")
            states = states.clone().detach().to(torch.float32).to("cuda") if torch.is_tensor(states) else torch.tensor(states, dtype=torch.float32).to("cuda"),
            states = states[0]

            # masks = [(~x.isnan()).float() for x in [a, b, c, d, e]]

            # print(states.shape, solo_labels.shape)
            solo_logits, dual_logits = model(states)
            # print(solo_logits.shape, dual_logits.shape, solo_labels.shape, dual_labels.shape)

            solo_loss = solo_loss_fn(
                solo_logits.view(-1, len(solo_encoder.classes_)),
                solo_labels.view(-1)
            )

            mask = torch.ones_like(dual_labels, dtype=torch.bool)
            mask[:, range(N), range(N)] = False

            dual_loss = dual_loss_fn(
                dual_logits[mask],
                dual_labels[mask]
            )

            solo_preds = solo_logits.argmax(dim=-1)       # (B, 4)
            dual_preds = dual_logits.argmax(dim=-1)       # (B, 4, 4)

            total_solo_loss += solo_loss.item()
            total_dual_loss += dual_loss.item()
            total_batches += 1
            
            solo_decoded = solo_encoder.inverse_transform(
                solo_preds.cpu().numpy().reshape(-1)
            ).reshape(solo_preds.shape)

            dual_decoded = dual_encoder.inverse_transform(
                dual_preds.cpu().numpy().reshape(-1)
            ).reshape(dual_preds.shape)

            decoded_df = extract_behavior_segments(solo_decoded, dual_decoded, chunk_start, video_id, test_df)
            decoded_dfs.append(decoded_df)

            decoded_labels_solo = solo_encoder.inverse_transform(
                solo_labels.cpu().numpy().reshape(-1)
            ).reshape(solo_labels.shape)
            decoded_labels_dual = dual_encoder.inverse_transform(
                dual_labels.cpu().numpy().reshape(-1)
            ).reshape(dual_labels.shape)

            actual_df = extract_behavior_segments(decoded_labels_solo, decoded_labels_dual, chunk_start, video_id, test_df)
            actual_dfs.append(actual_df)

    predicted = pd.concat(decoded_dfs, axis=0, ignore_index=True)
    labels = pd.concat(actual_dfs, axis=0, ignore_index=True)
    if predicted.empty:
        print("PREDICTED DF EMTPY. USING DUMMY ROW")
        dummy = {
        'lab_id': labels.iloc[0]['lab_id'],
        'video_id': labels.iloc[0]['video_id'],
        'agent_id': f'mouse1',
        'target_id': 'self',
        'action': 'rear',
        'behaviors_labeled': f'["mouse1,self,rear"]',
        'start_frame': 5,
        'stop_frame': 6,
        }
        predicted = pd.concat([predicted, pd.DataFrame([dummy])], ignore_index=True)
    score = mouse_fbeta(labels, predicted)
    print(f"############ SCOORE IS {score} ########### total dropped frames")

    model.train()
    # all_conf = torch.cat(all_conf)
    # all_has_action = torch.cat(all_has_action)
    # calculate_conf_acc(all_has_action, all_conf)

    # # Compute stats
    # conf_action_1 = all_conf[all_has_action == 1]
    # conf_action_0 = all_conf[all_has_action == 0]
    # stats_action_1 = describe(conf_action_1)
    # stats_action_0 = describe(conf_action_0)

    # print("Confidence stats for has_action == 1:")
    # print(stats_action_1)
    # print("\nConfidence stats for has_action == 0:")
    # print(stats_action_0)
    return total_solo_loss / total_batches, total_dual_loss / total_batches


def describe(tensor):
    return {
        'count': len(tensor),
        'min': tensor.min().item() if len(tensor) > 0 else None,
        'max': tensor.max().item() if len(tensor) > 0 else None,
        'mean': tensor.mean().item() if len(tensor) > 0 else None,
        'std': tensor.std().item() if len(tensor) > 0 else None
    }

def calculate_conf_acc(y_true, y_score, threshold=0.5):
    y_true = y_true.cpu().numpy() 
    y_score = y_score.cpu().numpy() 
    y_pred = (y_score >= threshold).astype(int)
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))

In [ ]:
for epoch in range(epochs):
    # Run evaluation and save every 5 epochs
    if (epoch + 1) % 1 == 0:
        test_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
        solo_eval_loss, dual_eval_loss = evaluate(model, test_loader, test_df)
        print(f"[Epoch {epoch+1}] Eval Solo Loss: {solo_eval_loss:.4f} | Eval Dual Loss: {dual_eval_loss:.4f}")

        model_path = f"umair_lesser_epoch{epoch+1}.pt"
        torch.save(model.state_dict(), model_path)
        print(f"✅ Saved model at {model_path}")
        
    model.train()

    total_solo_train_loss = 0
    total_dual_train_loss = 0
    total_conf_loss = 0
    total_train_batches = 0

    for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        if batch is None:
            continue
        states, solo_labels, dual_labels, has_action = batch
        states = states[0]
        dual_labels = dual_labels[0].to("cuda")
        solo_labels = solo_labels[0].to("cuda")
        has_action = has_action[0].to("cuda")

        states = states.clone().detach().to(torch.float32).to("cuda") if torch.is_tensor(states) else torch.tensor(states, dtype=torch.float32).to("cuda"),
        states = states[0]
        # print(a.shape, b.shape, c.shape, d.shape, e.shape, has_action.shape)
        # print(solo_labels.shape, dual_labels.shape)

        # masks = [(~x.isnan()).float() for x in [a, b, c, d, e]]

        solo_logits, dual_logits = model(states)

        solo_loss = solo_loss_fn(
            solo_logits.view(-1, len(solo_encoder.classes_)),
            solo_labels.view(-1)
        )

        mask = torch.ones_like(dual_labels, dtype=torch.bool)
        mask[:, range(N), range(N)] = False

        dual_loss = dual_loss_fn(
            dual_logits[mask],
            dual_labels[mask]
        )

        # print(has_action_conf.shape)
        # has_action_loss = has_action_loss_fn(
        #     has_action_conf.squeeze(),  # [B]
        #     has_action.float().squeeze()  # [B]
        # )
        # print("solo_logits:", torch.isnan(solo_logits).any(), torch.isinf(solo_logits).any())
        # print("dual_logits:", torch.isnan(dual_logits).any(), torch.isinf(dual_logits).any())
        # print("solo_labels:", torch.isnan(solo_labels).any(), torch.isinf(solo_labels).any())
        # print("dual_labels:", torch.isnan(dual_labels).any(), torch.isinf(dual_labels).any())

        total_loss = solo_loss + dual_loss# + (has_action_loss * ALPHA)
        # print(solo_loss, dual_loss, total_loss)
        # print(f"solo={solo_loss.item():.3f}, dual={dual_loss.item():.3f}, has_action={(ALPHA * has_action_loss.item()):.3f}, total={total_loss.item():.3f}")

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        total_dual_train_loss += dual_loss.item()
        total_solo_train_loss += solo_loss.item()
        # total_conf_loss += has_action_loss.item() * ALPHA
        total_train_batches += 1

    avg_solo_train_loss = total_solo_train_loss / total_train_batches
    avg_dual_train_loss = total_dual_train_loss / total_train_batches
    # avg_conf_loss = total_conf_loss / total_train_batches

    print(f"[Epoch {epoch+1}] "
          f"Train Solo Loss: {avg_solo_train_loss:.4f} | Train Dual Loss: {avg_dual_train_loss:.4f}")


  0%|          | 0/4175 [00:00<?, ?it/s]

############ SCOORE IS 0.010847299813780261 ########### total dropped frames
[Epoch 1] Eval Solo Loss: 2.1549 | Eval Dual Loss: 2.5318
✅ Saved model at umair_lesser_epoch1.pt


Epoch 1:   0%|          | 0/29088 [00:00<?, ?it/s]

[Epoch 1] Train Solo Loss: 0.7392 | Train Dual Loss: 1.1416


  0%|          | 0/4175 [00:00<?, ?it/s]

...

[Epoch 11] Train Solo Loss: 0.4332 | Train Dual Loss: 0.4344


  0%|          | 0/4175 [00:00<?, ?it/s]

############ SCOORE IS 0.16298794891036436 ########### total dropped frames
[Epoch 12] Eval Solo Loss: 0.5626 | Eval Dual Loss: 1.2733
✅ Saved model at umair_lesser_epoch12.pt


Epoch 12:   0%|          | 0/29088 [00:00<?, ?it/s]

returning None


In [31]:
test_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
solo_eval_loss, dual_eval_loss = evaluate(model, test_loader, test_df)
# print(f"[Epoch {epoch+1}] Eval Solo Loss: {solo_eval_loss:.4f} | Eval Dual Loss: {dual_eval_loss:.4f}")

# model_path = f"masked_cnn_attent_lesser_epoch{epoch+1}.pt"
# torch.save(model.state_dict(), model_path)
# print(f"✅ Saved model at {model_path}")

  0%|          | 0/4175 [00:00<?, ?it/s]


=== Metrics For This Lab TranquilPanther ===
  TP = 927
  FP = 2506
  FN = 2917
  Precision: 0.2700
  Recall:    0.2412
  Fβ:        0.2548
  Accuracy:  0.1460


=== Metrics For This Lab CalMS21_supplemental ===
  TP = 103319
  FP = 15497
  FN = 24528
  Precision: 0.8696
  Recall:    0.8081
  Fβ:        0.8377
  Accuracy:  0.7208


=== Metrics For This Lab CRIM13 ===
  TP = 1572
  FP = 2221
  FN = 8273
  Precision: 0.4144
  Recall:    0.1597
  Fβ:        0.2305
  Accuracy:  0.1303


=== Metrics For This Lab UppityFerret ===
  TP = 907
  FP = 328
  FN = 12205
  Precision: 0.7344
  Recall:    0.0692
  Fβ:        0.1264
  Accuracy:  0.0675


=== Metrics For This Lab CalMS21_task1 ===
  TP = 18215
  FP = 11619
  FN = 17797
  Precision: 0.6105
  Recall:    0.5058
  Fβ:        0.5533
  Accuracy:  0.3824


=== Metrics For This Lab BoisterousParrot ===
  TP = 0
  FP = 0
  FN = 6677
  Precision: 0.0000
  Recall:    0.0000
  Fβ:        0.0000
  Accuracy:  0.0000


=== Metrics For This Lab CalMS